In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("/content/comments toxicity.csv", encoding='latin1')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223549 entries, 0 to 223548
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             223549 non-null  object
 1   comment_text   223549 non-null  object
 2   toxic          223549 non-null  int64 
 3   severe_toxic   223549 non-null  int64 
 4   obscene        223549 non-null  int64 
 5   threat         223549 non-null  int64 
 6   insult         223549 non-null  int64 
 7   identity_hate  223549 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 13.6+ MB


In [ ]:
df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,223549.000000,223549.000000,223549.000000,223549.000000,223549.000000,223549.000000
mean,0.095657,0.008777,0.054306,0.003082,0.050566,0.009470
std,0.294121,0.093272,0.226621,0.055431,0.219110,0.096852
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
df.shape

(223549, 8)

In [ ]:
df.isnull().sum()

,0
id,0
comment_text,0
toxic,0
severe_toxic,0
obscene,0
threat,0
insult,0
identity_hate,0


In [ ]:
df.drop_duplicates(subset=['comment_text'], inplace=True)
display(f'shape after dropping duplicates: {df.shape}')

'shape after dropping duplicates: (223549, 8)'

In [ ]:
df.dropna(subset=['comment_text'], inplace=True)

In [ ]:
df['comment_text'] = df['comment_text'].str.lower()

In [ ]:
df['comment_text'] = df['comment_text'].str.strip()

In [ ]:
df['comment_text'] = df['comment_text'].str.replace(r'http\S+|www. \S+', '',regex=True)

In [ ]:
df = df[df['comment_text'] != '']

**TOKENIZATION**

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import nltk
nltk.download('punkt_tab')
df['tokens'] = df['comment_text'].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_467/2890994197.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tokens'] = df['comment_text'].apply(word_tokenize)


In [ ]:
df[['comment_text','tokens']].head(2)

,comment_text,tokens
0,explanation\nwhy the edits made under my usern...,"[explanation, why, the, edits, made, under, my..."
1,d'aww! he matches this background colour i'm s...,"[d'aww, !, he, matches, this, background, colo..."


**STOP WORD REMOVAL**

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

/tmp/ipykernel_467/3202537505.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])


**Lemmatization**

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
df['tokens'] = df['tokens'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

**Convert text into numbers**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)
df['comment_text'] = df['comment_text'].fillna('')
X = tfidf.fit_transform(df['comment_text'])
print("Shape:", X.shape)


Shape: (223542, 5000)


In [ ]:
toxicity_map = {'non-toxic': 0, 'toxic': 1}
df['label_toxicity'] = df['toxic']

In [ ]:
from sklearn.model_selection import train_test_split
y = df['label_toxicity']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
log_reg_model = LogisticRegression(max_iter=1000)
log_reg_model.fit(x_train, y_train)
print("Logistic Regression model trained successfully")

Logistic Regression model trained successfully


In [ ]:
y_predict = log_reg_model.predict(x_test)

In [ ]:
sample = ["you are a good boy "]
sample = tfidf.transform(sample)
print(log_reg_model.predict(sample))

[0]


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
print("Accuracy:", accuracy_score(y_test, y_predict))
print("precision:", precision_score(y_test, y_predict))
print("recall:", recall_score(y_test, y_predict))
print("f1 Score:", f1_score(y_test, y_predict))
print("confusion matrix:\n", confusion_matrix(y_test, y_predict))
print("classification report:\n", classification_report(y_test, y_predict))


Accuracy: 0.9516875796819433
precision: 0.8480968858131488
recall: 0.5874880153403643
f1 Score: 0.6941376380628717
confusion matrix:
 [[40098   439]
 [ 1721  2451]]
classification report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.97     40537
           1       0.85      0.59      0.69      4172

    accuracy                           0.95     44709
   macro avg       0.90      0.79      0.83     44709
weighted avg       0.95      0.95      0.95     44709

